# Unix Shell Tutorial: Filtering and Extracting Biomedical Data


This is the **third tutorial** in a series that will demonstrate how shell scripting can be used to perform the tasks that health and life science specialists may need to undertake to find and retrieve biomedical data and text. We will use the compound caffeine as an example and explore different public repositories to identify diseases related to it. The focus is not on the specific relationships we may discover, but on the process of obtaining them.

The objective of this tutorial is to learn how to efficiently filter and extract relevant data from the CSV file retrieved in the previous tutorial. Specifically, we will focus on filtering for proteins associated with putative caffeine-related diseases and extracting only the corresponding protein identifiers.

> This tutorial is part of a series of tutorials adapted as interactive versions of the hands-on steps described in the [Data and Text Processing for Health and Life Sciences](https://labs.rd.ciencias.ulisboa.pt/book/) book, which is licensed under the [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

## Step 1: Filtering Relevant Data with `grep`


Some data in the CSV file may not be relevant for our information need, so we may need to identify and extract only the relevant rows and columns. In this tutorial, we will first select relevant proteins (rows) using the command-line tool `grep`, and then select specific columns using the command-line tool `cut`.

Because our running example is caffeine, we will use a simple heuristic and keep only proteins whose identifiers include the species suffixes `_HUMAN`, `_RAT`, or `_MOUSE`.

Extracting lines from a text file is the main function of `grep`. Selection is performed by providing a pattern that `grep` searches for in each line and returning only matching lines. `grep` also supports more complex patterns such as regular expressions, which we will use later.

To get started, we first need to retrieve the data file generated in the previous tutorial. The following command downloads the `chebi_27732_xrefs_UniProt.csv` file directly from the GitHub repository:


In [1]:
%%bash
curl -s -O 'https://raw.githubusercontent.com/lasigeBioTM/data-text-processing-notebooks/refs/heads/main/data/chebi_27732_xrefs_UniProt.csv'

### Single and Multiple Patterns


We can execute the following command to select proteins that contain `_RAT` as the species suffix in the UniProt entry identifier (for example, `RYR1_RAT`):


In [2]:
%%bash
grep '_RAT' chebi_27732_xrefs_UniProt.csv


"27732","CHEBI:27732","chebi","RYR1_RAT","F1LMY4","uniprot"
"27732","CHEBI:27732","chebi","RYR2_RAT","B0LPN4","uniprot"


**Expected Output:** A shorter list of proteins, all containing `_RAT` as the species suffix in the UniProt entry identifier.


Note that, if instead we select proteins that contain `RAT` **without the underscore**, we get unwanted matches:

In [3]:
%%bash
grep 'RAT' chebi_27732_xrefs_UniProt.csv


"27732","CHEBI:27732","chebi","RYR1_RAT","F1LMY4","uniprot"
"27732","CHEBI:27732","chebi","PUP1_ARATH","Q9FZ96","uniprot"
"27732","CHEBI:27732","chebi","RYR2_RAT","B0LPN4","uniprot"


**Expected Output:** A list of proteins not only for `_RAT` but also for `_ARATH` (a small flowering plant, *Arabidopsis thaliana*), because `RAT` appears in both suffixes.


## Step 2: Multiple Pattern Matching


To use multiple patterns, we can repeat the `-e` option once per pattern:


In [4]:
%%bash
grep -e '_HUMAN' -e '_RAT' -e '_MOUSE' chebi_27732_xrefs_UniProt.csv


"27732","CHEBI:27732","chebi","RYR1_MOUSE","E9PZQ0","uniprot"
"27732","CHEBI:27732","chebi","RYR1_RAT","F1LMY4","uniprot"
"27732","CHEBI:27732","chebi","RYR1_HUMAN","P21817","uniprot"
"27732","CHEBI:27732","chebi","CP3A4_HUMAN","P08684","uniprot"
"27732","CHEBI:27732","chebi","ATR_HUMAN","Q13535","uniprot"
"27732","CHEBI:27732","chebi","SMG1_HUMAN","Q96Q15","uniprot"
"27732","CHEBI:27732","chebi","SMG1_MOUSE","Q8BKX6","uniprot"
"27732","CHEBI:27732","chebi","RYR2_HUMAN","Q92736","uniprot"
"27732","CHEBI:27732","chebi","RYR2_MOUSE","E9Q401","uniprot"
"27732","CHEBI:27732","chebi","PNKD_HUMAN","Q8N490","uniprot"
"27732","CHEBI:27732","chebi","ATR_MOUSE","Q9JKK8","uniprot"
"27732","CHEBI:27732","chebi","CP1A2_HUMAN","P05177","uniprot"
"27732","CHEBI:27732","chebi","RYR3_HUMAN","Q15413","uniprot"
"27732","CHEBI:27732","chebi","RYR2_RAT","B0LPN4","uniprot"
"27732","CHEBI:27732","chebi","RYR3_MOUSE","A2AGL3","uniprot"


**Expected Output:** A longer list of proteins matching any of the three species suffixes (`_HUMAN`, `_RAT`, or `_MOUSE`).

Tip: you can repeat `-e PATTERN` multiple times; `grep` will match a line if any of the patterns match.


In a terminal, we would typically use `| less` to scroll through long outputs. However, in a notebook environment, using `less` does not work well because it requires interactive input. Instead, we can use the `head` command to preview just the first few lines of the output. The `-n 10` option tells `head` to show only the first 10 lines.

In [5]:
%%bash
grep -e '_HUMAN' -e '_RAT' -e '_MOUSE' chebi_27732_xrefs_UniProt.csv | head -n 10


"27732","CHEBI:27732","chebi","RYR1_MOUSE","E9PZQ0","uniprot"
"27732","CHEBI:27732","chebi","RYR1_RAT","F1LMY4","uniprot"
"27732","CHEBI:27732","chebi","RYR1_HUMAN","P21817","uniprot"
"27732","CHEBI:27732","chebi","CP3A4_HUMAN","P08684","uniprot"
"27732","CHEBI:27732","chebi","ATR_HUMAN","Q13535","uniprot"
"27732","CHEBI:27732","chebi","SMG1_HUMAN","Q96Q15","uniprot"
"27732","CHEBI:27732","chebi","SMG1_MOUSE","Q8BKX6","uniprot"
"27732","CHEBI:27732","chebi","RYR2_HUMAN","Q92736","uniprot"
"27732","CHEBI:27732","chebi","RYR2_MOUSE","E9Q401","uniprot"
"27732","CHEBI:27732","chebi","PNKD_HUMAN","Q8N490","uniprot"


**Expected Output:** The first 10 lines of the filtered protein list.

## Creating and Updating the Script

We can now update our script file to contain the following lines:

```bash
url="https://www.ebi.ac.uk/ebisearch/ws/rest/chebi/entry/$1/xref/UniProtKB?size=100&format=csv"

curl -s "$url" | \
  grep -e '_HUMAN' -e '_RAT' -e '_MOUSE'
```

We added the `-s` option to suppress the progress information printed by `curl`. The trailing `\` characters indicate line continuation; they must be the last characters on the line (no trailing spaces).


In [6]:
%%bash
cat > getproteins.sh << 'EOF'
url="https://www.ebi.ac.uk/ebisearch/ws/rest/chebi/entry/$1/xref/UniProtKB?size=100&format=csv"

curl -s "$url" | \
  grep -e '_HUMAN' -e '_RAT' -e '_MOUSE'
EOF


**Expected Output:** Script file `getproteins.sh` created successfully.

In [7]:
%%bash
chmod u+x getproteins.sh

**Expected Output:** No output (permissions set successfully).

In [8]:
%%bash
./getproteins.sh 27732

"27732","CHEBI:27732","chebi","RYR1_MOUSE","E9PZQ0","uniprot"
"27732","CHEBI:27732","chebi","RYR1_RAT","F1LMY4","uniprot"
"27732","CHEBI:27732","chebi","RYR1_HUMAN","P21817","uniprot"
"27732","CHEBI:27732","chebi","CP3A4_HUMAN","P08684","uniprot"
"27732","CHEBI:27732","chebi","ATR_HUMAN","Q13535","uniprot"
"27732","CHEBI:27732","chebi","SMG1_HUMAN","Q96Q15","uniprot"
"27732","CHEBI:27732","chebi","SMG1_MOUSE","Q8BKX6","uniprot"
"27732","CHEBI:27732","chebi","RYR2_HUMAN","Q92736","uniprot"
"27732","CHEBI:27732","chebi","RYR2_MOUSE","E9Q401","uniprot"
"27732","CHEBI:27732","chebi","PNKD_HUMAN","Q8N490","uniprot"
"27732","CHEBI:27732","chebi","ATR_MOUSE","Q9JKK8","uniprot"
"27732","CHEBI:27732","chebi","CP1A2_HUMAN","P05177","uniprot"
"27732","CHEBI:27732","chebi","RYR3_HUMAN","Q15413","uniprot"
"27732","CHEBI:27732","chebi","RYR2_RAT","B0LPN4","uniprot"
"27732","CHEBI:27732","chebi","RYR3_MOUSE","A2AGL3","uniprot"


**Expected Output:** Filtered list of relevant proteins for caffeine (CHEBI:27732).

In [9]:
%%bash
./getproteins.sh 27732 > chebi_27732_xrefs_UniProt_relevant.csv

**Expected Output:** No output (file saved successfully).

## Step 3: Data Elements Selection with `cut`


Now we need to select specific columns from the CSV file. Selecting columns from a delimited text file is a common task for `cut`. The `cut` command can receive the delimiter character with `-d` and the fields (columns) to extract with `-f`.

In our CSV file, the **fifth** column contains the UniProt accession (for example, `P21817`). As a quick demonstration, the first column is the numeric ChEBI identifier (`27732`).


In [10]:
%%bash
cut -d, -f1 < chebi_27732_xrefs_UniProt_relevant.csv

"27732"
"27732"
"27732"
"27732"
"27732"
"27732"
"27732"
"27732"
"27732"
"27732"
"27732"
"27732"
"27732"
"27732"
"27732"


**Expected Output:** Only the first column of the file (the numeric ChEBI identifier).

In CSV files, commas (`,`) separate the columns. The command below prints only the first column.


We can also extract multiple columns at once by separating the column numbers with a comma. For example, to get both the first column (the ChEBI identifier) and the fifth column (the UniProt identifier), we use `-f1,5`:

In [11]:
%%bash
cut -d, -f1,5 < chebi_27732_xrefs_UniProt_relevant.csv

"27732","E9PZQ0"
"27732","F1LMY4"
"27732","P21817"
"27732","P08684"
"27732","Q13535"
"27732","Q96Q15"
"27732","Q8BKX6"
"27732","Q92736"
"27732","E9Q401"
"27732","Q8N490"
"27732","Q9JKK8"
"27732","P05177"
"27732","Q15413"
"27732","B0LPN4"
"27732","A2AGL3"


**Expected Output:** First and fifth columns of the file.

Now, the output contains both the first and fifth column of the file.

## Final Script with Column Selection

We can now update our script file to output **only** UniProt accessions (column 5). The script will:
1) retrieve the cross-references for a ChEBI identifier,
2) keep only `_HUMAN`, `_RAT`, and `_MOUSE` entries,
3) extract the UniProt accession column, and
4) remove quotation marks.

```bash
url="https://www.ebi.ac.uk/ebisearch/ws/rest/chebi/entry/$1/xref/UniProtKB?size=100&format=csv"

curl -s "$url" | \
  grep -e '_HUMAN' -e '_RAT' -e '_MOUSE' | \
  cut -d, -f5 | \
  tr -d '"'
```

The last two commands extract field 5 (the UniProt accession column) and remove the double-quote characters that appear in the CSV output.


In [12]:
%%bash
cat > getproteins.sh << 'EOF'
url="https://www.ebi.ac.uk/ebisearch/ws/rest/chebi/entry/$1/xref/UniProtKB?size=100&format=csv"

curl -s "$url" | \
  grep -e '_HUMAN' -e '_RAT' -e '_MOUSE' | \
  cut -d, -f5 | \
  tr -d '"'
EOF


**Expected Output:** Final script version created.

Now we can execute the script for caffeine:

In [13]:
%%bash
./getproteins.sh 27732

E9PZQ0
F1LMY4
P21817
P08684
Q13535
Q96Q15
Q8BKX6
Q92736
E9Q401
Q8N490
Q9JKK8
P05177
Q15413
B0LPN4
A2AGL3


**Expected Output:** Only protein identifiers (fifth column, with quotes removed) for proteins from HUMAN, RAT, and MOUSE species.

In [14]:
%%bash
./getproteins.sh 27732 > chebi_27732_xrefs_UniProt_relevant_identifiers.csv

**Expected Output:** No output (file saved successfully).

To check if the file was really created and to analyze its contents, we can
use the `cat` command:

In [15]:
%%bash
cat chebi_27732_xrefs_UniProt_relevant_identifiers.csv

E9PZQ0
F1LMY4
P21817
P08684
Q13535
Q96Q15
Q8BKX6
Q92736
E9Q401
Q8N490
Q9JKK8
P05177
Q15413
B0LPN4
A2AGL3


**Expected Output:** List of protein identifiers associated with caffeine-related diseases.

# Conclusion

This concludes the **Filtering and Extraction** tutorial adapted from the [Data and Text Processing for Health and Life Sciences](https://labs.rd.ciencias.ulisboa.pt/book/) book.

In this tutorial, we practiced filtering and extracting structured data using `grep`, `head`, `cut`, and `tr`, and we combined them into a small reusable shell script.

The next tutorial in this series will explore task repetition techniques to efficiently apply the same task to all proteins in the list we have gathered.


# Exercise

As an exercise, execute the script to extract only the protein identifiers associated with [water](https://www.ebi.ac.uk/chebi/searchId.do?chebiId=CHEBI:15377) and [gold](https://www.ebi.ac.uk/chebi/searchId.do?chebiId=CHEBI:30050).


In [16]:
%%bash
./getproteins.sh 15377 > water_proteins.csv

**Expected Output:** File `water_proteins.csv` created with protein identifiers for water (CHEBI:15377).

In [17]:
%%bash
./getproteins.sh 30050 > gold_proteins.csv

**Expected Output:** File `gold_proteins.csv` created with protein identifiers for gold (CHEBI:30050).

In [18]:
%%bash
cat water_proteins.csv

P36269
O55071
Q9EPW0


**Expected Output:** List of protein identifiers associated with water.

In [19]:
%%bash
cat gold_proteins.csv

Q96DZ5
Q63524
Q78IS1
Q9Y3A6
Q8WW62
Q8VDC1
Q9BQS8
Q15363
Q8R1V4
Q99KF1
Q5I0E7
Q5BK85
Q9R0Q3
Q9Y3Q3
Q6AY25
Q7Z7H5
Q9CXE7
Q6AXN3
Q9CQG0
Q86XR7
Q3UHI4
P49755
Q9Y3B3
D3ZTX0
Q6PL24
Q9BVK6
Q92503
Q7TNY6


**Expected Output:** List of protein identifiers associated with gold.